<a href="https://colab.research.google.com/github/sadeeshaa2002/drowsiness_ml/blob/main/drowsinessDetectionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Replace with your actual GitHub repository URL
!git clone https://github.com/sadeeshaa2002/drowsiness_ml.git
%cd drowsiness_ml
!ls

Cloning into 'drowsiness_ml'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 19 (delta 1), reused 19 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 4.65 MiB | 15.97 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/drowsiness_ml
convert_model.py     inspect_model.py  realtime_detector.py   train_model.py
evaluate_model.py    models	       test_onnx_realtime.py
extract_features.py  output	       test_realtime.py


In [3]:
!mv /content/drowsiness_dataset.csv /content/drowsiness_ml/

In [5]:
# Create the expected data subfolder
!mkdir -p data

# Move and rename your uploaded CSV to match train_model.py
!mv /content/drowsiness_dataset.csv ./data/drowsiness_features.csv 2>/dev/null || mv drowsiness_dataset.csv ./data/drowsiness_features.csv 2>/dev/null || true

In [7]:
import os
import pandas as pd

# 1. Print current location of files inside project folder
print("--- Project Files ---")
print(os.listdir('/content/drowsiness_ml'))

# 2. Check row count of drowsiness_dataset.csv if it exists
target_csv = '/content/drowsiness_ml/drowsiness_dataset.csv'
if os.path.exists(target_csv):
    df = pd.read_csv(target_csv)
    print(f"\n{target_csv} sample count: {len(df)}")
    print(df.head())
else:
    print(f"\n{target_csv} does not exist.")

--- Project Files ---
['.vscode', '.gitignore', '.git', 'test_onnx_realtime.py', 'inspect_model.py', 'models', 'output', 'realtime_detector.py', 'train_model.py', 'test_realtime.py', 'evaluate_model.py', 'data', 'extract_features.py', 'convert_model.py']

/content/drowsiness_ml/drowsiness_dataset.csv does not exist.


In [8]:
import pandas as pd
import shutil

# Copy dataset from output/ to the root drowsiness_ml directory
shutil.copy('/content/drowsiness_ml/output/drowsiness_dataset.csv', '/content/drowsiness_ml/drowsiness_dataset.csv')

# Verify the file is populated
df = pd.read_csv('/content/drowsiness_ml/drowsiness_dataset.csv')
print(f"Total samples in dataset: {len(df)}")
print(df.head())

Total samples in dataset: 0
Empty DataFrame
Columns: [EAR, MAR, Pitch, Yaw, Roll, Label]
Index: []


In [11]:
import os
import pandas as pd

for root, dirs, files in os.walk('/content/drowsiness_ml'):
    for file in files:
        if file.endswith('.csv'):
            path = os.path.join(root, file)
            size = os.path.getsize(path)
            try:
                df = pd.read_csv(path)
                rows = len(df)
            except Exception:
                rows = 0
            print(f"File: {path} | Size: {size} bytes | Rows: {rows}")

File: /content/drowsiness_ml/drowsiness_dataset.csv | Size: 29 bytes | Rows: 0
File: /content/drowsiness_ml/output/drowsiness_dataset.csv | Size: 29 bytes | Rows: 0
File: /content/drowsiness_ml/data/drowsiness_features.csv | Size: 30 bytes | Rows: 0


In [12]:
%cd /content/drowsiness_ml
!cp data/drowsiness_features.csv drowsiness_dataset.csv
!mkdir -p output
!cp data/drowsiness_features.csv output/drowsiness_dataset.csv

/content/drowsiness_ml


In [13]:
!python train_model.py

--- Loading Dataset ---
Dataset loaded: 41,775 samples.
Class Distribution:
 label
1    0.534746
0    0.465254
Name: proportion, dtype: float64

--- Training Balanced Random Forest Classifier ---

================ EVALUATION RESULTS ================
ROC-AUC Score: 0.9763

Classification Report:
               precision    recall  f1-score   support

       Alert       0.88      0.94      0.91      3887
      Drowsy       0.94      0.89      0.91      4468

    accuracy                           0.91      8355
   macro avg       0.91      0.91      0.91      8355
weighted avg       0.91      0.91      0.91      8355

Feature Importance Breakdown:
  EAR   : 30.80%
  MAR   : 10.91%
  PITCH : 15.09%
  YAW   : 21.91%
  ROLL  : 21.29%

SUCCESS! Optimized model saved to './output/drowsiness_model.pkl'


In [16]:
%%writefile /content/drowsiness_ml/evaluate_model.py
import os
import pandas as pd
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def evaluate():
    # Cross-platform path resolution
    csv_path = os.path.join("Data", "drowsiness_features.csv")
    model_path = os.path.join("output", "drowsiness_model.pkl")

    # Fallback paths for Colab/local directory differences
    if not os.path.exists(csv_path):
        if os.path.exists(os.path.join("data", "drowsiness_features.csv")):
            csv_path = os.path.join("data", "drowsiness_features.csv")
        elif os.path.exists("drowsiness_dataset.csv"):
            csv_path = "drowsiness_dataset.csv"
        else:
            raise FileNotFoundError(f"Dataset file not found at '{csv_path}'.")

    if not os.path.exists(model_path):
        if os.path.exists("drowsiness_model.pkl"):
            model_path = "drowsiness_model.pkl"
        else:
            raise FileNotFoundError(f"Model file not found at '{model_path}'.")

    print(f"--- Loading dataset from: {csv_path} ---")
    df = pd.read_csv(csv_path)

    X = df.drop(columns=["label"])
    y = df["label"]

    print(f"--- Loading model from: {model_path} ---")
    model = joblib.load(model_path)

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else None

    print("\n================ EVALUATION RESULTS ================")
    if y_prob is not None:
        print(f"ROC-AUC Score: {roc_auc_score(y, y_prob):.4f}")

    print("\nClassification Report:")
    print(classification_report(y, y_pred, target_names=["Alert", "Drowsy"]))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

if __name__ == "__main__":
    evaluate()

Overwriting /content/drowsiness_ml/evaluate_model.py


In [17]:
!python evaluate_model.py

--- Loading dataset from: Data/drowsiness_features.csv ---
--- Loading model from: output/drowsiness_model.pkl ---

================ EVALUATION RESULTS ================
ROC-AUC Score: 0.9862

Classification Report:
              precision    recall  f1-score   support

       Alert       0.89      0.96      0.93     19436
      Drowsy       0.96      0.90      0.93     22339

    accuracy                           0.93     41775
   macro avg       0.93      0.93      0.93     41775
weighted avg       0.93      0.93      0.93     41775


Confusion Matrix:
[[18689   747]
 [ 2266 20073]]


In [18]:
from google.colab import files
files.download('output/drowsiness_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>